## predict redshift using bliss & estimate

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import torch
import numpy as np
from os import environ
from pathlib import Path
from einops import rearrange
import pickle
from tqdm import tqdm 

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from hydra import initialize, compose
from hydra.utils import instantiate

from bliss.surveys.dc2 import DC2
from pathlib import Path
from sklearn.metrics import mean_squared_error

environ["BLISS_HOME"] = str(Path().resolve().parents[2])
home = environ["BLISS_HOME"]

output_dir = Path("./DC2_redshift_predict_output/")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
with initialize(config_path=f".", version_base=None):
    notebook_cfg = compose("notebook_predict_config")

In [ ]:
a = torch.randn((6))
a.shape

In [ ]:
b = torch.randn((1))
b.shape

In [ ]:
torch.cat((a,b), dim=0)

In [ ]:
torch.Tensor([]).shape

In [ ]:
torch.Tensor(1)[0].shape

In [ ]:
dc2: DC2 = instantiate(notebook_cfg.surveys.dc2)
dc2.setup()
dc2_train_dataset = dc2.train_dataset


# calculate mean of redshift in training set for baseline

In [ ]:
redshifts = []
for i in tqdm(range(len(dc2_train_dataset))):
    non_zero_indices = torch.nonzero(dc2_train_dataset[i]['tile_catalog']['redshifts'])
    redshifts += dc2_train_dataset[i]['tile_catalog']['redshifts'][non_zero_indices[:,0], non_zero_indices[:,1], non_zero_indices[:,2], non_zero_indices[:,3]].tolist()
np.mean(redshifts)

In [ ]:
true_redshifts = []
for i in tqdm(range(len(dc2.test_dataset))):
    non_zero_indices = torch.nonzero(dc2.test_dataset[i]['tile_catalog']['redshifts'])
    true_redshifts += dc2_train_dataset[i]['tile_catalog']['redshifts'][non_zero_indices[:,0], non_zero_indices[:,1], non_zero_indices[:,2], non_zero_indices[:,3]].tolist()

In [ ]:
from sklearn.metrics import mean_squared_error
mean_squared_error(true_redshifts, [1.1343084003841843] * len(true_redshifts))

In [ ]:
dc2_train_dataset[0]['tile_catalog']['locs'].sum()

In [ ]:
len(dc2_train_dataset)

In [ ]:
len(dc2.valid_dataset)

In [ ]:
len(dc2.test_dataset)

### Load One Full Image

In [ ]:
dc2: DC2 = instantiate(notebook_cfg.surveys.dc2)
test_sample = dc2.get_plotting_sample(0)

In [ ]:
# full image (4000 x 4000)
fig, ax = plt.subplots(figsize=(8, 8))
image = test_sample["image"][0]
ax.imshow(np.log((image - image.min()) + 10), cmap="viridis");

In [ ]:
cur_image_wcs = test_sample["wcs"]
cur_image_true_full_catalog = test_sample["full_catalog"]
cur_image_match_id = test_sample["match_id"]

In [ ]:
cur_image_true_full_catalog['redshifts'].shape

### Load the trained model and make prediction

In [ ]:
# change this model path according to your training setting
MODEL_PATH = f"{home}/output/DC2_redshift/DC2_redshift_only/checkpoints/encoder_val_loss=0.000000-v11.ckpt"
encoder = instantiate(notebook_cfg.encoder)
pretrained_weights = torch.load(MODEL_PATH)["state_dict"]
encoder.load_state_dict(pretrained_weights)
encoder.eval();

In [ ]:
# with open(f"{home}/case_studies/dc2/DC2_exp_output/plotting_model_output.pkl", "rb") as inputp: 
#     out_dict = pickle.load(inputp)

In [ ]:
batch = {
    "tile_catalog": test_sample["tile_catalog"],
    "images": rearrange(test_sample["image"], "h w nw -> 1 h w nw"),
    "background": rearrange(test_sample["background"], "h w nw -> 1 h w nw"),
    "psf_params": rearrange(test_sample["psf_params"], "h w -> 1 h w")
}

with torch.no_grad():
    out_dict = encoder.predict_step(batch, None)

with open(output_dir / "plotting_model_output.pkl", "wb") as outp:  # Overwrites any existing file.
    pickle.dump(out_dict, outp, pickle.HIGHEST_PROTOCOL)

In [ ]:
out_dict

In [ ]:
bliss_catalog = out_dict["mode_cat"]
bliss_catalog = bliss_catalog.to_full_catalog(4)
# bliss_catalog.plocs = bliss_catalog.plocs + 4
matcher = instantiate(notebook_cfg.encoder.matcher)
metrics = instantiate(notebook_cfg.encoder.metrics)
matching = matcher.match_catalogs(test_sample["full_catalog"], bliss_catalog)

In [ ]:
bliss_catalog["redshift"]

In [ ]:
matching

### Plotting

In [ ]:
# flux error
true_matches, bliss_matches = matching[0]
# true_flux = cur_image_true_full_catalog.on_nmgy[0, true_matches, :]
# bliss_flux = bliss_catalog.on_nmgy[0, bliss_matches, :]
true_redshift = cur_image_true_full_catalog["redshift"][0, true_matches, :]
bliss_redshift = bliss_catalog.data["redshift"][0, bliss_matches, :]
plt.scatter(
    x = np.array(true_redshift), 
    y = np.array(bliss_redshift), 
    alpha = 0.6,
    cmap = "viridis"
)
# plt.xlim(90, 1000)
plt.ylabel('Bliss redshift')
plt.xlabel('Truth redshift')
plt.axline((0, 0), slope=1, color='r', linestyle='--')
# plt.axhline(y=0, color='r', linestyle='--')
plt.show()

In [ ]:
true_flux

In [ ]:
len(bliss_redshift)

In [ ]:
mse = mean_squared_error(true_redshift, bliss_redshift)
print(f"mse:{mse:03f}")